In [5]:
import pandas as pd
import polars as pl

def get_reverse_complement(seq):
    """Get reverse complement of a sequence"""
    complement = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G', 'N': 'N'}
    return ''.join(complement[base] for base in seq[::-1].upper())

In [6]:
gtf_files = ['/data/reference/gencode.v47.basic.annotation.gtf.gz', '/data/reference/gencode.v47lift37.basic.annotation.gtf.gz']
for gtf_file in gtf_files:
    gtf = pl.read_csv(gtf_file, comment_prefix='#', separator='\t', has_header=False)
    gtf.columns = ['chrom', 'source', 'feature', 'start', 'end', 'score', 'strand', 'frame', 'attribute']
    # Extract gene_id, transcript_id, gene_name and exon_number from attribute column
    gtf = gtf.with_columns([
        pl.col('attribute').str.extract('gene_id "(.*?)"', 1).alias('gene_id'),
        pl.col('attribute').str.extract('transcript_id "(.*?)"', 1).alias('transcript_id'),
        pl.col('attribute').str.extract('gene_name "(.*?)"', 1).alias('gene_name'),
        pl.col('attribute').str.extract('gene_type "(.*?)"', 1).alias('gene_type'),
        pl.col('attribute').str.extract('transcript_type "(.*?)"', 1).alias('transcript_type'),
        pl.col('attribute').str.extract('exon_number (.*?);', 1).alias('exon_number')
    ])
    protein_coding_gtf = gtf.filter((pl.col('gene_type') == 'protein_coding') & (pl.col('feature') != 'gene'))
    protein_coding_gtf = protein_coding_gtf.with_columns(pl.col('start') - 1)
    # Group by transcript_id and get list of exon starts for each transcript
    exon_starts = protein_coding_gtf.filter(pl.col('feature') == 'CDS').group_by('transcript_id').agg(
        (pl.when(pl.col('strand').first() == '-')
        .then(
            pl.when(pl.col('start').count()==1)
            .then(
                (pl.col('start').first()-3).cast(str)
            )
            .otherwise(
                pl.concat_str([
                    (pl.col('start').sort().first() - 3).cast(str),
                    pl.col('start').sort().slice(1).cast(str).str.join(',')
                ], separator=',')
            )
        )
        .otherwise(
            pl.col('start').sort().cast(str).str.join(',')
        ) + ',').alias('exon_starts'),

        (pl.when(pl.col('strand').first() == '+')
        .then(
            pl.when(pl.col('end').count() == 1)
            .then(
                (pl.col('end').last()+3).cast(str)
            )
            .otherwise(
                pl.concat_str([
                    pl.col('end').sort().head(pl.col('end').count()-1).cast(str).str.join(','),
                    (pl.col('end').sort().last() + 3).cast(str)
                ], separator=',')
            )
        )
        .otherwise(
            pl.col('end').sort().cast(str).str.join(',')
        ) + ',').alias('exon_ends'),
        pl.col('exon_number').last().alias('exon_numbers')
    )
    cds_starts = protein_coding_gtf.filter(pl.col('feature') == 'CDS').group_by('transcript_id').agg(
        pl.when(pl.col('strand').first() == '-')
        .then(pl.col('start').last() - 3)
        .otherwise(pl.col('start').first())
        .alias('cds_starts'),
        pl.when(pl.col('strand').first() == '-')
        .then(pl.col('end').first())
        .otherwise(pl.col('end').last()+3)
        .alias('cds_ends'),
        # pl.col('exon_number').str.join(',').alias('cds_numbers')
    )
    tx_starts = protein_coding_gtf.filter(pl.col('feature') == 'transcript').group_by('transcript_id').agg(
        pl.col('gene_id').first().alias('gene_id'),
        pl.col('gene_name').first().alias('gene_name'),
        pl.col('chrom').first().alias('chrom'),
        pl.col('strand').first().alias('strand'),
        pl.col('start').min().alias('tx_starts'),
        pl.col('end').max().alias('tx_ends'),
        pl.col('transcript_type').first().alias('transcript_type'),
        # pl.col('exon_number').str.join(',').alias('tx_numbers')
    )
    # Join the three dataframes on transcript_id
    joined_df = tx_starts.join(cds_starts, on='transcript_id', how='inner')\
                        .join(exon_starts, on='transcript_id', how='inner')
    joined_df = joined_df.sort(['chrom', 'tx_starts'])
    joined_df = joined_df.select([
        'gene_id',
        'transcript_id',
        'chrom',
        'strand',
        'tx_starts',
        'tx_ends', 
        'cds_starts',
        'cds_ends',
        'exon_numbers',
        'exon_starts',
        'exon_ends',
        'gene_name'
    ]).rename({
        'transcript_id': 'name',
        'tx_starts': 'txStart',
        'tx_ends': 'txEnd',
        'cds_starts': 'cdsStart',
        'cds_ends': 'cdsEnd',
        'exon_starts': 'exonStarts',
        'exon_ends': 'exonEnds'
    })
    output_file = gtf_file.replace('.gtf.gz', '.processed.tsv')
    joined_df.write_csv(output_file, separator='\t')

    output_file = gtf_file.replace('.gtf.gz', '.transcripts.bed')

    protein_coding_gtf.filter(pl.col('feature')=='transcript')\
            .select(['chrom','start','end','transcript_id'])\
            .write_csv(output_file, separator='\t', include_header=False)

    output_file = gtf_file.replace('.gtf.gz', '.exons.bed')
    protein_coding_gtf.filter(pl.col('feature')=='exon')\
            .with_columns((pl.col('transcript_id') + 'exon' + pl.col('exon_number')).alias('exon_name'))\
            .select(['chrom','start','end','exon_name'])\
            .write_csv(output_file, separator='\t', include_header=False)

## quality control of CDS

In [8]:
import polars as pl
ann = pl.read_csv('/data/reference/gencode.v47.basic.annotation.processed.tsv', separator='\t')
# Remove duplicates based on CDS and exon information
print(ann.shape)
ann = ann.unique(subset=['chrom','strand','cdsStart', 'cdsEnd', 'exonStarts', 'exonEnds'])
print(ann.shape)
ann.head()



(65049, 12)
(52138, 12)


gene_id,name,chrom,strand,txStart,txEnd,cdsStart,cdsEnd,exon_numbers,exonStarts,exonEnds,gene_name
str,str,str,str,i64,i64,i64,i64,i64,str,str,str
"""ENSG00000197858.12""","""ENST00000361036.11""","""chr8""","""+""",144082588,144086216,144082730,144086125,11,"""144082730,144083388,144083713,…","""144082804,144083500,144083861,…","""GPAA1"""
"""ENSG00000105372.8""","""ENST00000221975.6""","""chr19""","""+""",41860256,41871416,41869080,41871377,6,"""41869080,41869698,41871350,""","""41869214,41869753,41871377,""","""RPS19"""
"""ENSG00000225973.4""","""ENST00000436697.3""","""chr15""","""-""",55317183,55319123,55317627,55317792,2,"""55317627,""","""55317792,""","""PIGBOS1"""
"""ENSG00000084676.16""","""ENST00000348332.8""","""chr2""","""+""",24491253,24770702,24658677,24768391,23,"""24658677,24665748,24673365,246…","""24658766,24665915,24673463,246…","""NCOA1"""
"""ENSG00000128973.13""","""ENST00000637494.1""","""chr15""","""-""",68208115,68229677,68208139,68229584,5,"""68208139,68209636,68211262,682…","""68208410,68209759,68211318,682…","""CLN6"""


In [9]:
import pyfaidx
fasta = {}

with pyfaidx.Fasta('/data/reference/hg38/hg38.fa') as f:
    for chrom in ann['chrom'].unique():
        fasta[chrom] = f[chrom][:].seq

In [10]:
def extract_cds_sequence(row, fasta):
    """Extract CDS sequence for a transcript based on exon coordinates and CDS boundaries."""
    chrom = row['chrom']
    strand = row['strand']
    cds_start = row['cdsStart']
    cds_end = row['cdsEnd']
    
    # Parse exon coordinates
    exon_starts = [int(x) for x in row['exonStarts'].rstrip(',').split(',')]
    exon_ends = [int(x) for x in row['exonEnds'].rstrip(',').split(',')]
    
    # Extract CDS sequence from exons
    cds_sequence = ""
    
    for start, end in zip(exon_starts, exon_ends):
        # Find overlap between exon and CDS
        overlap_start = max(start, cds_start)
        overlap_end = min(end, cds_end)
        
        if overlap_start < overlap_end:
            # Extract sequence from this exon segment
            seq = str(fasta[chrom][overlap_start:overlap_end]).upper()
            cds_sequence += seq
    
    # Reverse complement if on negative strand
    if strand == '-':
        cds_sequence = get_reverse_complement(cds_sequence)
    
    return cds_sequence

# Extract CDS sequences for all transcripts
sequences = []
for row in ann.iter_rows(named=True):
    seq = extract_cds_sequence(row, fasta)
    sequences.append(seq)

# Add sequences to the dataframe
ann = ann.with_columns(pl.Series("cds_sequence", sequences))
ann.head()


gene_id,name,chrom,strand,txStart,txEnd,cdsStart,cdsEnd,exon_numbers,exonStarts,exonEnds,gene_name,cds_sequence
str,str,str,str,i64,i64,i64,i64,i64,str,str,str,str
"""ENSG00000197858.12""","""ENST00000361036.11""","""chr8""","""+""",144082588,144086216,144082730,144086125,11,"""144082730,144083388,144083713,…","""144082804,144083500,144083861,…","""GPAA1""","""ATGGGCCTCCTGTCGGACCCGGTTCGCCGG…"
"""ENSG00000105372.8""","""ENST00000221975.6""","""chr19""","""+""",41860256,41871416,41869080,41871377,6,"""41869080,41869698,41871350,""","""41869214,41869753,41871377,""","""RPS19""","""ATGACCAAGATCTATGGGGGACGTCAGAGA…"
"""ENSG00000225973.4""","""ENST00000436697.3""","""chr15""","""-""",55317183,55319123,55317627,55317792,2,"""55317627,""","""55317792,""","""PIGBOS1""","""ATGTTTAGGAGATTGACTTTTGCACAACTG…"
"""ENSG00000084676.16""","""ENST00000348332.8""","""chr2""","""+""",24491253,24770702,24658677,24768391,23,"""24658677,24665748,24673365,246…","""24658766,24665915,24673463,246…","""NCOA1""","""ATGAGTGGCCTCGGGGACAGTTCATCCGAC…"
"""ENSG00000128973.13""","""ENST00000637494.1""","""chr15""","""-""",68208115,68229677,68208139,68229584,5,"""68208139,68209636,68211262,682…","""68208410,68209759,68211318,682…","""CLN6""","""ATGGAGGCGACGCGGAGGCGGCAGCACCTG…"


In [11]:
# Check CDS sequences for start codons, stop codons, and length divisibility by 3
def check_cds_quality(sequence):
    """Check if CDS sequence has proper start/stop codons, length divisible by 3, and no internal stop codons."""
    if not sequence or len(sequence) < 3:
        return {
            'has_start_codon': False,
            'has_stop_codon': False,
            'length_divisible_by_3': False,
            'has_internal_stop_codons': False,
            'length': len(sequence) if sequence else 0
        }
    
    has_start_codon = sequence[:3] == 'ATG'
    has_stop_codon = sequence[-3:] in ['TAA', 'TAG', 'TGA']
    length_divisible_by_3 = len(sequence) % 3 == 0
    
    # Check for internal stop codons (excluding the last codon)
    has_internal_stop_codons = False
    if len(sequence) >= 6:  # Need at least 2 codons to check for internal stops
        # Check all codons except the last one
        for i in range(0, len(sequence) - 3, 3):
            codon = sequence[i:i+3]
            if codon in ['TAA', 'TAG', 'TGA']:
                has_internal_stop_codons = True
                break
    
    return {
        'has_start_codon': has_start_codon,
        'has_stop_codon': has_stop_codon,
        'length_divisible_by_3': length_divisible_by_3,
        'has_internal_stop_codons': has_internal_stop_codons,
        'length': len(sequence)
    }

# Apply quality checks to all CDS sequences
quality_checks = []
for row in ann.iter_rows(named=True):
    seq = row['cds_sequence']
    quality = check_cds_quality(seq)
    quality_checks.append(quality)

# Add quality check columns to the dataframe
ann = ann.with_columns([
    pl.Series("has_start_codon", [q['has_start_codon'] for q in quality_checks]),
    pl.Series("has_stop_codon", [q['has_stop_codon'] for q in quality_checks]),
    pl.Series("length_divisible_by_3", [q['length_divisible_by_3'] for q in quality_checks]),
    pl.Series("has_internal_stop_codons", [q['has_internal_stop_codons'] for q in quality_checks]),
    pl.Series("cds_length", [q['length'] for q in quality_checks])
])

# Display summary statistics before filtering
print("CDS Quality Summary (before filtering):")
print(f"Total transcripts: {len(ann)}")
print(f"Has start codon (ATG): {ann['has_start_codon'].sum()} ({ann['has_start_codon'].mean()*100:.1f}%)")
print(f"Has stop codon (TAA/TAG/TGA): {ann['has_stop_codon'].sum()} ({ann['has_stop_codon'].mean()*100:.1f}%)")
print(f"Length divisible by 3: {ann['length_divisible_by_3'].sum()} ({ann['length_divisible_by_3'].mean()*100:.1f}%)")
print(f"Has internal stop codons: {ann['has_internal_stop_codons'].sum()} ({ann['has_internal_stop_codons'].mean()*100:.1f}%)")
print(f"All quality criteria met (no internal stops): {(ann['has_start_codon'] & ann['has_stop_codon'] & ann['length_divisible_by_3'] & ~ann['has_internal_stop_codons']).sum()}")

# Filter to keep only rows that pass all quality criteria
ann = ann.filter(
    ann['has_start_codon'] & 
    ann['has_stop_codon'] & 
    ann['length_divisible_by_3'] & 
    ~ann['has_internal_stop_codons']
)

print(f"\nAfter filtering: {len(ann)} transcripts remaining")

# Show some examples
print("\nFirst few transcripts with quality checks:")


CDS Quality Summary (before filtering):
Total transcripts: 52138
Has start codon (ATG): 51793 (99.3%)
Has stop codon (TAA/TAG/TGA): 51789 (99.3%)
Length divisible by 3: 51958 (99.7%)
Has internal stop codons: 244 (0.5%)
All quality criteria met (no internal stops): 51357

After filtering: 51357 transcripts remaining

First few transcripts with quality checks:


name,gene_name,cds_length,has_start_codon,has_stop_codon,length_divisible_by_3,has_internal_stop_codons
str,str,i64,bool,bool,bool,bool
"""ENST00000361036.11""","""GPAA1""",1686,true,true,true,false
"""ENST00000221975.6""","""RPS19""",216,true,true,true,false
"""ENST00000436697.3""","""PIGBOS1""",165,true,true,true,false
"""ENST00000348332.8""","""NCOA1""",4326,true,true,true,false
"""ENST00000637494.1""","""CLN6""",648,true,true,true,false
"""ENST00000295754.10""","""TGFBR2""",1704,true,true,true,false
"""ENST00000587352.5""","""ANKRD27""",1170,true,true,true,false
"""ENST00000584943.1""","""ODF4""",429,true,true,true,false
"""ENST00000640574.1""","""GABRG2""",873,true,true,true,false


In [16]:
ann.filter(~pl.col('has_start_codon'))

gene_id,name,chrom,strand,txStart,txEnd,cdsStart,cdsEnd,exon_numbers,exonStarts,exonEnds,gene_name,cds_sequence,has_start_codon,has_stop_codon,length_divisible_by_3,has_internal_stop_codons,cds_length
str,str,str,str,i64,i64,i64,i64,i64,str,str,str,str,bool,bool,bool,bool,i64


In [13]:
# Remove duplicate CDS sequences
initial_count = len(ann)
ann = ann.unique(subset=['cds_sequence'], keep='first')
ann = ann.sort(['chrom', 'txStart'])
print(f"Removed {initial_count - len(ann)} duplicate CDS sequences ({len(ann)} remaining)")

Removed 242 duplicate CDS sequences (51115 remaining)


In [14]:
ann.write_csv('/data/reference/gencode.v47.basic.annotation.processed.filtered.tsv', separator='\t')

## Generate all possible variants

In [15]:
import polars as pl
import numpy as np
from tqdm import tqdm
ann = pl.read_csv('/data/reference/gencode.v47.basic.annotation.processed.filtered.tsv', separator='\t')

# Generate all possible variants at each coding sequence position
variants = []

# Define complement mapping for negative strand
complement = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G'}

# Genetic code table
genetic_code = {
    'TTT': 'F', 'TTC': 'F', 'TTA': 'L', 'TTG': 'L',
    'TCT': 'S', 'TCC': 'S', 'TCA': 'S', 'TCG': 'S',
    'TAT': 'Y', 'TAC': 'Y', 'TAA': '*', 'TAG': '*',
    'TGT': 'C', 'TGC': 'C', 'TGA': '*', 'TGG': 'W',
    'CTT': 'L', 'CTC': 'L', 'CTA': 'L', 'CTG': 'L',
    'CCT': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',
    'CAT': 'H', 'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q',
    'CGT': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R',
    'ATT': 'I', 'ATC': 'I', 'ATA': 'I', 'ATG': 'M',
    'ACT': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
    'AAT': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K',
    'AGT': 'S', 'AGC': 'S', 'AGA': 'R', 'AGG': 'R',
    'GTT': 'V', 'GTC': 'V', 'GTA': 'V', 'GTG': 'V',
    'GCT': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
    'GAT': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E',
    'GGT': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G'
}

for row in tqdm(ann.iter_rows(named=True), total=len(ann), desc="Processing transcripts"):
    cds_seq = row['cds_sequence']
    transcript_id = row['name']
    gene_name = row['gene_name']
    chrom = row['chrom']
    strand = row['strand']
    
    # Parse exon coordinates
    exon_starts = [int(x) for x in row['exonStarts'].rstrip(',').split(',')]
    exon_ends = [int(x) for x in row['exonEnds'].rstrip(',').split(',')]
    cds_start = row['cdsStart']
    cds_end = row['cdsEnd']
    
    # Build mapping from CDS position to genomic coordinate
    cds_to_genomic = {}
    cds_pos = 0
    
    # Create list of CDS genomic positions in order
    cds_genomic_positions = []
    
    for i, (start, end) in enumerate(zip(exon_starts, exon_ends)):
        # Find overlap with CDS region
        overlap_start = max(start, cds_start)
        overlap_end = min(end, cds_end)
        
        if overlap_start < overlap_end:
            # This exon contains CDS sequence
            for genomic_pos in range(overlap_start, overlap_end):
                cds_genomic_positions.append(genomic_pos)
    
    # For negative strand, reverse the genomic positions to match CDS sequence order
    if strand == '-':
        cds_genomic_positions.reverse()
    
    # Map CDS positions to genomic coordinates
    for cds_pos, genomic_pos in enumerate(cds_genomic_positions):
        cds_to_genomic[cds_pos] = genomic_pos
    
    # For each position in the CDS
    for pos in range(len(cds_seq)):
        if pos in cds_to_genomic:
            genomic_coord = cds_to_genomic[pos] + 1  # Convert to 1-based
            original_base = cds_seq[pos].upper()
            
            # Convert to positive strand representation
            if strand == '-':
                # For negative strand, convert bases to their complement for positive strand representation
                pos_strand_original = complement[original_base]
                pos_strand_alts = [complement[base] for base in ['A', 'T', 'G', 'C'] if complement[base] != pos_strand_original]
            else:
                # For positive strand, use bases as-is
                pos_strand_original = original_base
                pos_strand_alts = [base for base in ['A', 'T', 'G', 'C'] if base != original_base]
            
            # Get codon position and original codon
            codon_start = (pos // 3) * 3
            codon_pos_in_codon = pos % 3
            
            # Skip if we can't form a complete codon
            if codon_start + 2 >= len(cds_seq):
                continue
            
            # Skip start codon (first codon) and stop codon (last codon)
            codon_number = pos // 3
            total_codons = len(cds_seq) // 3
            if codon_number == 0 or codon_number == total_codons - 1:
                continue
                
            original_codon = cds_seq[codon_start:codon_start+3].upper()
            original_aa = genetic_code.get(original_codon, 'X')
            
            # Generate all possible substitutions
            for alt_base in pos_strand_alts:
                # Create mutated codon (in transcript orientation)
                mutated_base_transcript = complement[alt_base] if strand == '-' else alt_base
                mutated_codon = original_codon[:codon_pos_in_codon] + mutated_base_transcript + original_codon[codon_pos_in_codon+1:]
                mutated_aa = genetic_code.get(mutated_codon, 'X')
                
                # Only keep missense variants (amino acid changes, excluding stop codons and original stop codons)
                if original_aa != mutated_aa and original_aa != '*' and mutated_aa != '*':
                    variants.append({
                        'transcript_id': transcript_id,
                        'gene_name': gene_name,
                        'chrom': chrom,
                        'genomic_pos': genomic_coord,
                        'strand': strand,
                        'var_rel_pos_in_cds': pos,  # 0-based position
                        'ref': pos_strand_original,  # Always in positive strand
                        'alt': alt_base,  # Always in positive strand
                        'ref_codon': original_codon,
                        'alt_codon': mutated_codon,
                    })

# Convert to polars DataFrame
variants_df = pl.DataFrame(variants)
print(f"Generated {len(variants_df)} missense variants across {len(ann)} transcripts")
variants_df.head()

Processing transcripts: 100%|██████████| 51115/51115 [03:58<00:00, 214.17it/s]


Generated 187385237 missense variants across 51115 transcripts


transcript_id,gene_name,chrom,genomic_pos,strand,var_rel_pos_in_cds,ref,alt,ref_codon,alt_codon
str,str,str,i64,str,i64,str,str,str,str
"""ENST00000641515.2""","""OR4F5""","""chr1""",65568,"""+""",3,"""A""","""G""","""AAG""","""GAG"""
"""ENST00000641515.2""","""OR4F5""","""chr1""",65568,"""+""",3,"""A""","""C""","""AAG""","""CAG"""
"""ENST00000641515.2""","""OR4F5""","""chr1""",65569,"""+""",4,"""A""","""T""","""AAG""","""ATG"""
"""ENST00000641515.2""","""OR4F5""","""chr1""",65569,"""+""",4,"""A""","""G""","""AAG""","""AGG"""
"""ENST00000641515.2""","""OR4F5""","""chr1""",65569,"""+""",4,"""A""","""C""","""AAG""","""ACG"""


In [16]:
variants_df.write_csv('/data/reference/gencode.v47.basic.annotation.processed.filtered.variants_with_ann.tsv', separator='\t')

In [17]:
print(f"Number of unique variants: {variants_df.select(['chrom', 'genomic_pos', 'ref', 'alt']).n_unique()}")
variants_df.head()

Number of unique variants: 75760985


transcript_id,gene_name,chrom,genomic_pos,strand,var_rel_pos_in_cds,ref,alt,ref_codon,alt_codon
str,str,str,i64,str,i64,str,str,str,str
"""ENST00000641515.2""","""OR4F5""","""chr1""",65568,"""+""",3,"""A""","""G""","""AAG""","""GAG"""
"""ENST00000641515.2""","""OR4F5""","""chr1""",65568,"""+""",3,"""A""","""C""","""AAG""","""CAG"""
"""ENST00000641515.2""","""OR4F5""","""chr1""",65569,"""+""",4,"""A""","""T""","""AAG""","""ATG"""
"""ENST00000641515.2""","""OR4F5""","""chr1""",65569,"""+""",4,"""A""","""G""","""AAG""","""AGG"""
"""ENST00000641515.2""","""OR4F5""","""chr1""",65569,"""+""",4,"""A""","""C""","""AAG""","""ACG"""


In [18]:
variants_df.select(['chrom', 'genomic_pos', 'ref', 'alt']).unique().write_csv('/data/reference/gencode.v47.basic.annotation.processed.filtered.variants.tsv', separator='\t')

## Cleaning the variants

In [19]:
import polars as pl
valid_chroms = [f'chr{i}' for i in range(1, 23)] + ['chrX', 'chrY']
variants = pl.read_csv('/data/reference/gencode.v47.basic.annotation.processed.filtered.variants.tsv', separator='\t')
variants = variants.sort(['chrom', 'genomic_pos', 'ref', 'alt'])
variants = variants.filter(pl.col('chrom').is_in(valid_chroms))

In [20]:
gnomad_exomes = []
for chrom in variants['chrom'].unique():
    gnomad_exomes.append(pl.read_csv(f'/data/gnomad.exomes.v4.1/{chrom}.tsv.gz', separator='\t', has_header=False))
gnomad_exomes = pl.concat(gnomad_exomes)
gnomad_exomes.columns = ['chrom', 'pos', 'ref', 'alt', 'af', 'ac', 'an']
gnomad_exomes = gnomad_exomes.sort(['chrom', 'pos'])
gnomad_exomes.head()


chrom,pos,ref,alt,af,ac,an
str,i64,str,str,f64,i64,i64
"""chr1""",12541,"""C""","""G""",0.000032,1,30958
"""chr1""",12591,"""C""","""T""",0.000101,5,49392
"""chr1""",12592,"""C""","""T""",0.000101,5,49676
"""chr1""",12990,"""A""","""G""",0.0001,20,200184
"""chr1""",12992,"""G""","""A""",0.000059,12,202300


In [21]:
gnomad_genomes = []
for chrom in variants['chrom'].unique():
    gnomad_genomes.append(pl.read_csv(f'/data/gnomad.genomes.v4.1/{chrom}.tsv.gz', separator='\t', has_header=False))
gnomad_genomes = pl.concat(gnomad_genomes)
gnomad_genomes.columns = ['chrom', 'pos', 'ref', 'alt', 'af', 'ac', 'an']
gnomad_genomes = gnomad_genomes.sort(['chrom', 'pos'])
gnomad_genomes.head()


chrom,pos,ref,alt,af,ac,an
str,i64,str,str,f64,i64,i64
"""chr1""",10111,"""C""","""A""",0.000023,1,44330
"""chr1""",10131,"""C""","""A""",0.000009,1,105956
"""chr1""",10139,"""A""","""T""",0.000017,1,58784
"""chr1""",10140,"""A""","""C""",0.000027,2,72760
"""chr1""",10141,"""C""","""G""",0.00002,1,50842


In [22]:
# Find variants that are not in gnomad_exomes
variants_not_in_gnomad = variants.join(
    gnomad_exomes.select(['chrom', 'pos', 'ref', 'alt']),
    left_on=['chrom', 'genomic_pos', 'ref', 'alt'],
    right_on=['chrom', 'pos', 'ref', 'alt'],
    how='anti'
).join(
    gnomad_genomes.select(['chrom', 'pos', 'ref', 'alt']),
    left_on=['chrom', 'genomic_pos', 'ref', 'alt'],
    right_on=['chrom', 'pos', 'ref', 'alt'],
    how='anti'
)

print(f"Total variants: {len(variants)}")
print(f"Variants not in gnomAD exomes: {len(variants_not_in_gnomad)}")
print(f"Percentage not in gnomAD: {len(variants_not_in_gnomad) / len(variants) * 100:.2f}%")

variants_not_in_gnomad.head()


Total variants: 75760380
Variants not in gnomAD exomes: 62027306
Percentage not in gnomAD: 81.87%


chrom,genomic_pos,ref,alt
str,i64,str,str
"""chr1""",65568,"""A""","""C"""
"""chr1""",65568,"""A""","""G"""
"""chr1""",65569,"""A""","""C"""
"""chr1""",65569,"""A""","""G"""
"""chr1""",65569,"""A""","""T"""


In [23]:
gnomad_exomes['an'].max()

1461894

In [24]:
common_variants = [gnomad_exomes.filter((pl.col('af') > 0.0005) & (pl.col('an') > pl.col('an').max() * 0.1)),
                   gnomad_genomes.filter((pl.col('af') > 0.0005) & (pl.col('an') > pl.col('an').max() * 0.1))]
common_variants = pl.concat(common_variants).unique(subset=['chrom', 'pos', 'ref', 'alt'])
# Find variants that are common (present in gnomAD with AF > 0.001)
common_variants_in_our_set = variants.join(
    common_variants.select(['chrom', 'pos', 'ref', 'alt']),
    left_on=['chrom', 'genomic_pos', 'ref', 'alt'],
    right_on=['chrom', 'pos', 'ref', 'alt'],
    how='inner'
)

print(f"Total variants: {len(variants)}")
print(f"Common variants in our set: {len(common_variants_in_our_set)}")
print(f"Percentage of common variants: {len(common_variants_in_our_set) / len(variants) * 100:.2f}%")

common_variants_in_our_set.head()




Total variants: 75760380
Common variants in our set: 162082
Percentage of common variants: 0.21%


chrom,genomic_pos,ref,alt
str,i64,str,str
"""chr1""",69428,"""T""","""G"""
"""chr1""",69496,"""G""","""A"""
"""chr1""",69511,"""A""","""G"""
"""chr1""",69590,"""T""","""A"""
"""chr1""",69640,"""G""","""C"""


In [25]:
common_variants_in_our_set.write_csv('/data/reference/gencode.v47.basic.annotation.processed.filtered.variants.common.tsv', separator='\t')
variants_not_in_gnomad.write_csv('/data/reference/gencode.v47.basic.annotation.processed.filtered.variants.not_in_gnomad.tsv', separator='\t')

## Sample with matched trinucletide context

In [26]:
import pyfaidx

fasta = {}
with pyfaidx.Fasta('/data/reference/hg38/hg38.fa') as f:
    for chrom in f.keys():
        fasta[chrom] = f[chrom][:].seq


In [27]:
import polars as pl

common_variants = pl.read_csv('/data/reference/gencode.v47.basic.annotation.processed.filtered.variants.common.tsv', separator='\t')
unseen_variants = pl.read_csv('/data/reference/gencode.v47.basic.annotation.processed.filtered.variants.not_in_gnomad.tsv', separator='\t')


In [28]:
def get_trinucleotide_context(chrom, pos, ref, alt, fasta):
    """Get trinucleotide context for a variant"""
    start = pos - 2
    end = pos + 1
    trinuc = fasta[chrom][start:end].upper()
    return trinuc if trinuc[1] == ref else None



def add_trinuc_context(df):
    """Add trinucleotide context to variants dataframe"""
    return df.with_columns([
        pl.struct(['chrom', 'genomic_pos', 'ref', 'alt']).map_elements(
            lambda x: get_trinucleotide_context(x['chrom'], x['genomic_pos'], x['ref'], x['alt'], fasta),
            return_dtype=pl.Utf8
        ).alias('trinuc_context')
    ]).filter(pl.col('trinuc_context').is_not_null()).with_columns([
        pl.when(pl.col('ref').is_in(['A', 'G']))
        .then(pl.col('trinuc_context').map_elements(get_reverse_complement, return_dtype=pl.Utf8))
        .otherwise(pl.col('trinuc_context'))
        .alias('trinuc_context_std'),
        
        pl.when(pl.col('ref').is_in(['A', 'G']))
        .then(pl.col('ref').map_elements(get_reverse_complement, return_dtype=pl.Utf8))
        .otherwise(pl.col('ref'))
        .alias('ref_std'),
        
        pl.when(pl.col('ref').is_in(['A', 'G']))
        .then(pl.col('alt').map_elements(get_reverse_complement, return_dtype=pl.Utf8))
        .otherwise(pl.col('alt'))
        .alias('alt_std')
    ])

# Add trinucleotide context to both datasets
common_variants_with_context = add_trinuc_context(common_variants)
unseen_variants_with_context = add_trinuc_context(unseen_variants)

common_variants_with_context.head()

chrom,genomic_pos,ref,alt,trinuc_context,trinuc_context_std,ref_std,alt_std
str,i64,str,str,str,str,str,str
"""chr1""",69428,"""T""","""G""","""TTT""","""TTT""","""T""","""G"""
"""chr1""",69496,"""G""","""A""","""CGG""","""CCG""","""C""","""T"""
"""chr1""",69511,"""A""","""G""","""CAC""","""GTG""","""T""","""C"""
"""chr1""",69590,"""T""","""A""","""GTC""","""GTC""","""T""","""A"""
"""chr1""",69640,"""G""","""C""","""AGA""","""TCT""","""C""","""G"""


In [29]:
common_variants_with_context.write_csv('/data/reference/gencode.v47.basic.annotation.processed.filtered.variants.common.trinuc.tsv', separator='\t')
unseen_variants_with_context.write_csv('/data/reference/gencode.v47.basic.annotation.processed.filtered.variants.not_in_gnomad.trinuc.tsv', separator='\t')

In [33]:
common_variants_with_context = pl.read_csv('/data/reference/gencode.v47.basic.annotation.processed.filtered.variants.common.trinuc.tsv', separator='\t')
unseen_variants_with_context = pl.read_csv('/data/reference/gencode.v47.basic.annotation.processed.filtered.variants.not_in_gnomad.trinuc.tsv', separator='\t')

In [34]:
common_variants_with_context

chrom,genomic_pos,ref,alt,trinuc_context,trinuc_context_std,ref_std,alt_std
str,i64,str,str,str,str,str,str
"""chr1""",69428,"""T""","""G""","""TTT""","""TTT""","""T""","""G"""
"""chr1""",69496,"""G""","""A""","""CGG""","""CCG""","""C""","""T"""
"""chr1""",69511,"""A""","""G""","""CAC""","""GTG""","""T""","""C"""
"""chr1""",69590,"""T""","""A""","""GTC""","""GTC""","""T""","""A"""
"""chr1""",69640,"""G""","""C""","""AGA""","""TCT""","""C""","""G"""
…,…,…,…,…,…,…,…
"""chrY""",19706182,"""C""","""T""","""TCG""","""TCG""","""C""","""T"""
"""chrY""",19708037,"""C""","""T""","""GCT""","""GCT""","""C""","""T"""
"""chrY""",19715668,"""G""","""C""","""GGT""","""ACC""","""C""","""G"""


In [ ]:
# Helper function to merge variants with transcripts and sample one per variant
def get_variants_with_transcripts(variants, variants_df, ann):
    return (variants.join(variants_df, on=['chrom', 'genomic_pos', 'ref', 'alt'])
            .group_by(['chrom', 'genomic_pos', 'ref', 'alt'])
            .agg(pl.all().sample(1).first())
            .join(ann.select(['name', 'gene_id', 'cds_sequence']), 
                  left_on='transcript_id', right_on='name', how='inner')
            .with_columns((pl.col('var_rel_pos_in_cds') // 3).alias('codon_pos')))

# Process common variants
common_final = get_variants_with_transcripts(common_variants_with_context, variants_df, ann)

# Get mutation type counts from common variants
mutation_counts = common_variants_with_context.group_by(
    ['trinuc_context_std', 'ref_std', 'alt_std']
).len().sort(['trinuc_context_std', 'ref_std', 'alt_std'])

# Sample matching variants from unseen dataset
sampled_unseen = []
for row in mutation_counts.iter_rows(named=True):
    matching = unseen_variants_with_context.filter(
        (pl.col('trinuc_context_std') == row['trinuc_context_std']) &
        (pl.col('ref_std') == row['ref_std']) & 
        (pl.col('alt_std') == row['alt_std'])
    )
    sampled_unseen.append(matching.sample(min(row['len'], len(matching)), seed=42))

# Process matched unseen variants
matched_unseen_final = get_variants_with_transcripts(
    pl.concat(sampled_unseen), variants_df, ann
) if sampled_unseen else None

print(f"Common variants: {len(common_final)}")
print(f"Matched unseen variants: {len(matched_unseen_final) if matched_unseen_final is not None else 0}")

common_final.head()


## cleaning

In [1]:
import sys
from pathlib import Path
import polars as pl
import json
from tqdm import tqdm

def remove_transcript_in_exclusion_regions(transcript_data, exclusion_regions_path=None):
        # Add MHC region to exclude_regions
    mhc_region = pl.DataFrame({
        'chrom': ['chr6'],
        'start': [28510120-1],
        'end': [33480577]
    })
    if exclusion_regions_path is not None:
        exclude_regions = pl.read_csv(exclusion_regions_path, separator='\t', has_header=False)
        exclude_regions.columns = ['chrom', 'start', 'end']
        exclude_regions = pl.concat([exclude_regions, mhc_region])
    else:
        exclude_regions = mhc_region
    # Merge overlapping regions
    exclude_regions = exclude_regions.sort(['chrom', 'start'])
    merged_regions = []
    current_region = None

    for row in exclude_regions.iter_rows(named=True):
        if current_region is None:
            current_region = row
        elif (row['chrom'] == current_region['chrom'] and 
            row['start'] <= current_region['end']):
            # Regions overlap - merge them
            current_region = {
                'chrom': current_region['chrom'],
                'start': current_region['start'], 
                'end': max(current_region['end'], row['end'])
            }
        else:
            # No overlap - add current region and start new one
            merged_regions.append(current_region)
            current_region = row

    if current_region is not None:
        merged_regions.append(current_region)

    exclude_regions = pl.DataFrame(merged_regions)

    transcript_data = transcript_data.sort(['chrom', 'txStart'])
    exclude_regions = exclude_regions.sort(['chrom', 'start'])
    overlaps = [False] * len(transcript_data)
    print(transcript_data.shape[0])
    i = j = 0
    while i < len(transcript_data) and j < len(exclude_regions):
        if transcript_data['chrom'][i] == exclude_regions['chrom'][j]:
            if max(transcript_data['txStart'][i], exclude_regions['start'][j]) < min(transcript_data['txEnd'][i], exclude_regions['end'][j]):
                overlaps[i] = True
                i += 1
            elif transcript_data['txEnd'][i] < exclude_regions['start'][j]:
                i += 1
            else:
                j += 1
        elif transcript_data['chrom'][i] < exclude_regions['chrom'][j]:
            i += 1
        else:
            j += 1
    transcript_data = transcript_data.filter(~pl.Series(overlaps))

    return transcript_data

transcript_data_path = '/data/reference/gencode.v47.basic.annotation.processed.filtered.tsv'
transcript_data = pl.read_csv(transcript_data_path, separator='\t')
transcript_data = remove_transcript_in_exclusion_regions(transcript_data)
transcript_data.write_csv('/data/reference/gencode.v47.basic.annotation.processed.filtered.no_exclusion_regions.tsv', separator='\t')

## These code were used for missense variant finetuning. Skip for now

In [85]:
import sys
import os
from pathlib import Path
import polars as pl
import json
from tqdm import tqdm
import time
import torch
from typing import Callable

class TranscriptDataset(torch.utils.data.Dataset):
    def __init__(self, common_data_path, unseen_data_path, all_variants_path, transcript_data_path, 
                 tokenizer, 
                process_item: Callable = lambda *x, **kwargs: (x, kwargs),
                  exclusion_regions_path=None,
                 context_length=2048,
                 ignore_index=-100):
        self.context_length = context_length
        self.ignore_index = ignore_index
        self.tokenizer = tokenizer
        self.process_item = process_item

        self.transcript_data_path = Path(transcript_data_path)
        self.common_data_path = Path(common_data_path)
        self.unseen_data_path = Path(unseen_data_path)
        self.all_variants_path = Path(all_variants_path)
        self.exclusion_regions_path = Path(exclusion_regions_path) if exclusion_regions_path else None

        self.transcript_data = pl.read_csv(transcript_data_path, separator='\t')
        self.transcript_data = remove_transcript_in_exclusion_regions(self.transcript_data, self.exclusion_regions_path)

        self.common_variants = pl.read_csv(common_data_path, separator='\t')
        self.unseen_variants = pl.read_csv(unseen_data_path, separator='\t')
        self.all_variants = pl.read_csv(all_variants_path, separator='\t')
        # Remove variants that don't have a matching transcript in transcript_data
        transcript_ids = self.transcript_data['id'].to_list()
        self.all_variants = self.all_variants.filter(pl.col('tx_idx').is_in(transcript_ids))

        self.get_tokenized_sequences()
        self.matched_unseen = self.sample_unseen_variants()
        self.common_variants = self.common_variants.with_columns(pl.lit(0).alias('label'))
        self.matched_unseen = self.matched_unseen.with_columns(pl.lit(1).alias('label'))
        self.variants = self.all_variants.join(pl.concat([self.common_variants, self.matched_unseen]), 
                                               on=['chrom', 'genomic_pos', 'ref', 'alt'], how='inner')\
                                         .with_columns((pl.col('var_rel_pos_in_cds') // 3).alias('codon_pos'))

    def get_tokenized_sequences(self):
        tokenized_cache_path = self.transcript_data_path.with_suffix('.tokenized.json')
        if tokenized_cache_path.exists():
            print(f"Found cached tokenized sequences at {tokenized_cache_path}")
            with open(tokenized_cache_path, 'r') as f:
                temp_tokenized = json.load(f)
            self.tokenized = {int(k): v for k, v in temp_tokenized.items()}
        else:
            print(f"No cached tokens found. Tokenizing sequences and saving to {tokenized_cache_path}...")
            self.tokenized = {}
            for row_id, cds_seq in tqdm(self.transcript_data.select(['id', 'cds_sequence']).iter_rows(), total=self.transcript_data.height, desc="Tokenizing Transcripts"):
                if cds_seq is not None:
                    tokens = self.tokenizer.convert_tokens_to_ids(self.tokenizer.tokenize(cds_seq))
                    self.tokenized[row_id] = tokens
                else:
                    self.tokenized[row_id] = [] 
            with open(tokenized_cache_path, 'w') as f:
                json.dump(self.tokenized, f)

    def reset_unseen_variants(self):
        self.matched_unseen = self.sample_unseen_variants()
        self.matched_unseen = self.matched_unseen.with_columns(pl.lit(1).alias('label'))
        self.variants = self.all_variants.join(pl.concat([self.common_variants, self.matched_unseen]), 
                                               on=['chrom', 'genomic_pos', 'ref', 'alt'], how='inner') \
                                         .with_columns((pl.col('var_rel_pos_in_cds') // 3).alias('codon_pos'))



    def sample_unseen_variants(self, seed=None):
        # Get mutation type counts from common variants
        mutation_counts = self.common_variants.group_by(
            ['trinuc_context_std', 'ref_std', 'alt_std']
        ).len().sort(['trinuc_context_std', 'ref_std', 'alt_std'])

        if seed is None:
            seed = int(time.time())
        
        # Sample matching variants from unseen dataset
        sampled_unseen = []
        for row in mutation_counts.iter_rows(named=True):
            matching = self.unseen_variants.filter(
                (pl.col('trinuc_context_std') == row['trinuc_context_std']) &
                (pl.col('ref_std') == row['ref_std']) & 
                (pl.col('alt_std') == row['alt_std'])
            )
            sampled_unseen.append(matching.sample(min(row['len'], len(matching)), seed=42))

        # Process matched unseen variants
        matched_unseen = pl.concat(sampled_unseen)
        return matched_unseen

    def __len__(self):
        return len(self.transcript_data)

    def __getitem__(self, idx):
        tx_idx = self.transcript_data[idx,'id']
        variants = self.variants.filter(pl.col('tx_idx') == tx_idx)
        transcript_seq = self.tokenized[tx_idx]
        return self.process_item(transcript_seq, variants,
                                 tokenizer=self.tokenizer,
                                 context_length=self.context_length)


In [91]:


def process_item(tx_ids, tx_variants, tokenizer, context_length):
    tx_ids = np.array(tx_ids[:context_length-2])
    # - add CLS token
    tx_ids = np.insert(tx_ids, 0, tokenizer.cls_token_id)
    # - add SEP token
    tx_ids = np.append(tx_ids, tokenizer.sep_token_id)

    tx_variants = tx_variants.filter(pl.col('codon_pos') < context_length-2)
    pos_to_mask = tx_variants['codon_pos'].unique().to_numpy()
    pos_to_mask.sort()
    tx_ids[pos_to_mask+1] = tokenizer.mask_token_id

    mask = np.zeros(len(tx_ids), dtype=np.uint8)
    mask[pos_to_mask+1] = 1

    variant_pos = tx_variants['codon_pos'].to_numpy() + 1
    ref_codon_ids = tx_variants['ref_codon_id'].to_numpy()
    alt_codon_ids = tx_variants['alt_codon_id'].to_numpy()
    labels = tx_variants['label'].to_numpy()
    # labels = np.zeros((len(tx_ids), n_codons), dtype=np.int32) + ignore_index
    # labels[tx_variants['codon_pos'].to_numpy() + 1, tx_variants['ref_codon_id'].to_numpy()] = 0
    # labels[tx_variants['codon_pos'].to_numpy() + 1, tx_variants['alt_codon_id'].to_numpy()] = tx_variants['label'].to_numpy()
    # print(labels.shape, len(tx_ids))
    attention_mask = np.zeros(len(tx_ids), dtype=np.int32)

    padding_length = context_length - len(tx_ids)
    if padding_length > 0:
        tx_ids = np.pad(tx_ids, (0, padding_length),
                                        constant_values=tokenizer.pad_token_id)
        mask = np.pad(mask, (0, padding_length), constant_values=0)
        attention_mask = np.pad(attention_mask, (0, padding_length), constant_values=1)
        # labels = np.pad(labels, ((0, padding_length), (0,0)), constant_values=ignore_index)

    attention_mask = ~(attention_mask.astype(bool))
    mask = mask.astype(bool)
    output = {
        'input_ids': tx_ids,
        'labels': labels,
        'variant_pos': variant_pos,
        'ref_codon_ids': ref_codon_ids,
        'alt_codon_ids': alt_codon_ids,
        "attention_mask": attention_mask,
        "mask": mask
    }
    return output

In [87]:
sys.path.append('/workspace/codon_fm/')
from src.tokenizer import Tokenizer
tokenizer = Tokenizer(
            cls_token="<CLS>",
            bos_token="<CLS>",
            sep_token="<SEP>",
            unk_token="<UNK>",
            pad_token="<PAD>",
            mask_token="<MASK>",
            padding_side="right",
            truncation="right",
            seq_type="dna",
        )
common_data_path = '/data/reference/gencode.v47.basic.annotation.processed.filtered.variants.common.trinuc.tsv'
unseen_data_path = '/data/reference/gencode.v47.basic.annotation.processed.filtered.variants.not_in_gnomad.trinuc.tsv'
all_variants_path = '/data/reference/gencode.v47.basic.annotation.processed.filtered.variants_with_ann.tsv'
transcript_data_path = '/data/reference/gencode.v47.basic.annotation.processed.filtered.no_exclusion_regions.tsv'

In [92]:
transcript_dataset = TranscriptDataset(
    common_data_path=common_data_path,
    unseen_data_path=unseen_data_path,
    all_variants_path=all_variants_path,
    transcript_data_path=transcript_data_path,
    tokenizer=tokenizer,
    process_item=process_item,
)

46773
Found cached tokenized sequences at /data/reference/gencode.v47.basic.annotation.processed.filtered.no_exclusion_regions.tokenized.json


In [93]:
transcript_dataset[0]

{'input_ids': array([ 0, 19,  7, ...,  3,  3,  3]),
 'labels': array([1, 1, 1, 1, 0, 0, 0, 0, 0, 1], dtype=int32),
 'variant_pos': array([  6,  15,  18,  61, 134, 157, 162, 188, 205, 310]),
 'ref_codon_ids': array([41, 11, 11, 17, 68, 46,  9, 50, 40, 11]),
 'alt_codon_ids': array([ 9, 19, 19, 49, 64, 14, 41, 38, 24, 19]),
 'attention_mask': array([ True,  True,  True, ..., False, False, False]),
 'mask': array([False, False, False, ..., False, False, False])}

In [63]:
common_data_path = '/data/reference/gencode.v47.basic.annotation.processed.filtered.variants.common.trinuc.tsv'
unseen_data_path = '/data/reference/gencode.v47.basic.annotation.processed.filtered.variants.not_in_gnomad.trinuc.tsv'
all_variants_path = '/data/reference/gencode.v47.basic.annotation.processed.filtered.variants_with_ann.tsv'
transcript_data_path = '/data/reference/gencode.v47.basic.annotation.processed.filtered.tsv'
context_length = 2048

import sys
sys.path.append('/workspace/codon_fm/')
from src.tokenizer import Tokenizer
tokenizer = Tokenizer(
            cls_token="<CLS>",
            bos_token="<CLS>",
            sep_token="<SEP>",
            unk_token="<UNK>",
            pad_token="<PAD>",
            mask_token="<MASK>",
            padding_side="right",
            truncation="right",
            seq_type="dna",
        )

variant_dataset = VariantDataset(
    benign_data_path=common_data_path,
    unseen_data_path=unseen_data_path,
    all_variants_path=all_variants_path,
    transcript_data_path=transcript_data_path,
    tokenizer=tokenizer,
)




Loading transcripts...
Loading transcripts took 0.03 seconds
Found cached tokenized sequences
Tokenization took 0.94 seconds
Loading variants...
Loading variants took 1.59 seconds
Filtering variants took 0.68 seconds
Loading common variants took 0.00 seconds
Creating common variants for splitting took 11.64 seconds
Loading unseen variants took 0.40 seconds
Creating unseen variants pool took 3.63 seconds
Splitting data took 0.01 seconds
Total data loading time: 18.93 seconds
